In [ ]:
pip install -q pylatexenc matplotlib qiskit[visualization]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.5 MB/s eta 0:00:00


In [ ]:
pip install qiskit

In [ ]:
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler
from qiskit.visualization import plot_histogram

# 1. Initialize 2 qubits: Qubit 0 = Atom, Qubit 1 = Cat
qc = QuantumCircuit(2, 2)

# 2. Put the atom into a 50/50 superposition (decayed vs. undecayed)
qc.h(0)

# 3. Entangle the cat with the atom (if decayed (|1>), the poison triggers and kills the cat (|1>))
qc.cx(0, 1)

# 4. Open the box (measure both qubits)
qc.measure([0, 1], [0, 1])

# Display the circuit
print("Schrödinger's Cat Circuit:")
print(qc.draw(output="text"))

# 5. Run the simulation
sampler = StatevectorSampler()
job = sampler.run([qc], shots=1000)
result = job.result()
counts = result[0].data.c.get_counts()

print("\nMeasurement Results (Shots = 1000):")
for outcome, count in counts.items():
    # In Qiskit little-endian format: outcome = 'cat atom'
    status = "Alive Cat (Undecayed Atom)" if outcome == "00" else "Dead Cat (Decayed Atom)"
    print(f"|{outcome}> : {count} times -> {status}")

Schrödinger's Cat Circuit:
     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 

Measurement Results (Shots = 1000):
|11> : 498 times -> Dead Cat (Decayed Atom)
|00> : 502 times -> Alive Cat (Undecayed Atom)


In [ ]:
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler


def create_ghz_cat_state(n_qubits: int) -> QuantumCircuit:
    """Generates an N-qubit GHZ / Macroscopic Cat State."""
    qc = QuantumCircuit(n_qubits, n_qubits)

    # 1. Superposition on the control qubit (Atom)
    qc.h(0)

    # 2. Cascade CNOTs to entangle all remaining qubits (Cat environment)
    for i in range(n_qubits - 1):
        qc.cx(i, i + 1)

    # 3. Measurement (Open the box)
    qc.measure_all(add_bits=False)

    return qc


# Example: 5-qubit Cat State
n = 5
ghz_circuit = create_ghz_cat_state(n)

print(f"{n}-Qubit Schrödinger Cat (GHZ) Circuit:")
print(ghz_circuit.draw(output="text"))

# Run execution with StatevectorSampler
sampler = StatevectorSampler()
job = sampler.run([ghz_circuit], shots=1024)
result = job.result()
counts = result[0].data.c.get_counts()

print("\nMeasurement Counts:")
print(counts)

5-Qubit Schrödinger Cat (GHZ) Circuit:
     ┌───┐                     ░ ┌─┐            
q_0: ┤ H ├──■──────────────────░─┤M├────────────
     └───┘┌─┴─┐                ░ └╥┘┌─┐         
q_1: ─────┤ X ├──■─────────────░──╫─┤M├─────────
          └───┘┌─┴─┐           ░  ║ └╥┘┌─┐      
q_2: ──────────┤ X ├──■────────░──╫──╫─┤M├──────
               └───┘┌─┴─┐      ░  ║  ║ └╥┘┌─┐   
q_3: ───────────────┤ X ├──■───░──╫──╫──╫─┤M├───
                    └───┘┌─┴─┐ ░  ║  ║  ║ └╥┘┌─┐
q_4: ────────────────────┤ X ├─░──╫──╫──╫──╫─┤M├
                         └───┘ ░  ║  ║  ║  ║ └╥┘
c: 5/═════════════════════════════╩══╩══╩══╩══╩═
                                  0  1  2  3  4 

Measurement Counts:
{'11111': 532, '00000': 492}


Alternative:

Logarithmic-Depth (O(logN)) Tree Construction
For large N on hardware, a linear CNOT chain (O(N) depth) accumulates decoherence error along the chain. A binary tree structure creates the GHZ state in O(logN) circuit depth:

In [ ]:
def create_ghz_log_depth(n_qubits: int) -> QuantumCircuit:
    qc = QuantumCircuit(n_qubits)
    qc.h(0)

    step = 1
    while step < n_qubits:
        for i in range(0, n_qubits - step, 2 * step):
            target = i + step
            if target < n_qubits:
                qc.cx(i, target)
        step *= 2

    return qc

In [ ]:
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_bloch_multivector, plot_state_qsphere

# 1. Build the Schrödinger's Cat / Bell state without measurements
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

# 2. Extract the exact analytical statevector
state = Statevector.from_instruction(qc)
print("Analytical Statevector:")
print(state)

# 3. Plot the Q-Sphere (Best for multi-qubit entangled states)
fig_qsphere = plot_state_qsphere(state)
plt.show()

# 4. Plot individual Bloch Multivectors (Illustrates entanglement nuance)
fig_bloch = plot_bloch_multivector(state)
plt.show()

Analytical Statevector:
Statevector([0.70710678+0.j, 0.        +0.j, 0.        +0.j,
             0.70710678+0.j],
            dims=(2, 2))


In [2]:
pip install pennylane

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 96.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 83.1 MB/s eta 0:00:00
  Attempting uninstall: autograd
    Found existing installation: autograd 1.9.1
    Uninstalling autograd-1.9.1:
      Successfully uninstalled autograd-1.9.1


Pennylane Version

In [3]:
import pennylane as qml

# 1. Define device (default.qubit simulator with 2 wires)
dev = qml.device("default.qubit", wires=2, shots=1000)


@qml.qnode(dev)
def schrodinger_cat_circuit():
    # Put atom (wire 0) into 50/50 superposition
    qml.Hadamard(wires=0)

    # Entangle cat (wire 1) with the atom
    qml.CNOT(wires=[0, 1])

    # Sample measurement outcomes
    return qml.counts()


# Execute circuit
counts = schrodinger_cat_circuit()

print("PennyLane Execution Results:")
for state, count in counts.items():
    status = "Alive Cat" if state == "00" else "Dead Cat"
    print(f"|{state}> : {count} shots -> {status}")

# Draw the circuit
drawer = qml.draw(schrodinger_cat_circuit)()
print("\nCircuit Diagram:\n", drawer)

PennyLane Execution Results:
|00> : 521 shots -> Alive Cat
|11> : 479 shots -> Dead Cat

Circuit Diagram:
 0: ──H─╭●─┤  Counts
1: ────╰X─┤  Counts


/usr/local/lib/python3.12/dist-packages/pennylane/devices/device_api.py:207: PennyLaneDeprecationWarning: Setting shots on device is deprecated. Please use the `set_shots` transform on the respective QNode instead.
  warnings.warn(


Approach 1:
Pass shots to @qml.qnode (Recommended)
Remove shots=1000 from qml.device() and place it in the decorator:

In [6]:
import pennylane as qml

# 1. Initialize device without shots
dev = qml.device("default.qubit", wires=2)


# 2. Specify shots directly on the QNode
@qml.qnode(dev, shots=1000)
def schrodinger_cat_circuit():
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0, 1])
    return qml.counts()

# Execute circuit
counts = schrodinger_cat_circuit()

print("PennyLane Execution Results:")
for state, count in counts.items():
    status = "Alive Cat" if state == "00" else "Dead Cat"
    print(f"|{state}> : {count} shots -> {status}")

# Draw the circuit
drawer = qml.draw(schrodinger_cat_circuit)()
print("\nCircuit Diagram:\n", drawer)

PennyLane Execution Results:
|00> : 499 shots -> Alive Cat
|11> : 501 shots -> Dead Cat

Circuit Diagram:
 0: ──H─╭●─┤  Counts
1: ────╰X─┤  Counts


Approach2:
Pass shots Dynamically at Call Time
You can also specify shots when invoking the function:

In [8]:
import pennylane as qml

# 1. Device without shots (modern PennyLane device API)
dev = qml.device("default.qubit", wires=2)


# 2. Set shots directly in the @qml.qnode decorator
@qml.qnode(dev, shots=1000)
def schrodinger_cat_circuit():
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0, 1])
    return qml.counts()


# 3. Execute without passing shots as an argument
counts = schrodinger_cat_circuit()

print("PennyLane Execution Results:")
for state, count in counts.items():
    status = "Alive Cat" if state == "00" else "Dead Cat"
    print(f"|{state}> : {count} shots -> {status}")

# 4. Draw the circuit
print("\nCircuit Diagram:")
print(qml.draw(schrodinger_cat_circuit)())

PennyLane Execution Results:
|00> : 475 shots -> Alive Cat
|11> : 525 shots -> Dead Cat

Circuit Diagram:
0: ──H─╭●─┤  Counts
1: ────╰X─┤  Counts
